Задание 1. Определите подходящую модель БД для каждого модуля

| Модуль | Требования | Предпочтительная модель | Почему? |
| :--- | :--- | :--- | :--- |
| **Профиль пользователя** | Персональные данные, навыки, история активности | **Документноориентированная** | У профилей часто гибкая схема (разное количество навыков и типов активности), также документноориентированная модель отлично справляется с вложенными списками. |
| **Каталог курсов** | Курсы, уроки, авторы, категории | **Документоориентированная** |Курсы имеют сложную вложенную структуру (модули, уроки, материалы). |
| **Прогресс обучения** | Отметки о пройденных уроках, тестах | **Ключ-значение** | Модель "ключ-значение" обеспечивает хорошую скорость и отлично подойдет для быстрых операций чтения/записи по простому ключу |
| **Обсуждения** | Комментарии к урокам | **Документоориентированная** | Комментарии часто образуют иерархию (дерево ответов), которую удобно хранить в виде документов для быстрой загрузки веток. |
| **Поиск** | Поиск по курсам, пользователям, тегам | **Колоночная** | Колоночные СУБД способны невероятно быстро сканировать огромные массивы данных по отдельным столбцам. Это позволяет фильтровать курсы и пользователей по множеству тегов, категорий и атрибутов с хорошей скоростью. |
| **Рекомендации** | На основе активности и интересов | **Графовая**  | Идеально подходит для анализа сложных связей. Мгновенно вычисляет паттерны вроде «люди с похожими навыками проходили этот курс». |
| **Аналитика** | Статистика, дашборды, метрики | **Колоночная** | Позволяет очень быстро агрегировать данные (суммы, средние значения) по миллионам строк для построения графиков в реальном времени. |

Задание 2. Обоснование выбора через CAP-теорему 

| Модуль | CAP-выбор (CP/AP) | Обоснование |
| :--- | :--- | :--- |
| **Авторизация** | **CP** (Согласованность) | Если права пользователя изменились (например, доступ закрыт), это должно мгновенно отразиться на всех уcтройствах. Лучше отказать в доступе из-за недоступности сервиса (пожертвовать Availability), чем пустить пользователя с неактуальными правами (нарушить Consistency). |
| **Прогресс обучения** | **AP** (Доступность) | Для пользователя важнее непрерывность процесса обучения. Своевременность прогресса обучения не критично, главное — не блокировать интерфейс и дать пользователю учиться дальше. |
| **Аналитика** | **AP** (Доступность) | Для аналитических дашбордов вполне нормально показывать данные с небольшой задержкой. Строгая согласованность здесь избыточна, а высокая доступность на чтение — более приоритетна. |

Задание 3. Архитектурное решение — мультимодельность?
Может ли быть полезным использование нескольких типов БД в одном проекте?
Ответьте на вопросы:
- Какие модели можно комбинировать?
- Какие преимущества даёт такой подход?
- Есть ли недостатки (например, сложность синхронизации)?

Да, можно использовать разные модели. В одном проекте (особенно при микросервисной архитектуре) можно комбинировать их все, подбирая идеальный инструмент под конкретный сервис.Примеры связок:
* Реляционная + Ключ-значение: Основные данные (юзеры, транзакции) лежат в SQL, а Key-Value кэширует тяжелые запросы и хранит сессии.
* Документоориентированная + Реляционная: Контент и структура уроков (гибкий JSON) живут в документной БД, а прогресс и оценки (строгие связи) — в реляционной.
* Графовая + Колоночная: Графовая БД быстро строит рекомендации на лету, в то время как колоночная в фоне собирает кликстрим и логи этих рекомендаций для аналитики.

Преимущества подхода:
* Оптимальная производительность: Каждая БД решает только ту задачу, для которой была спроектирована.
* Независимое масштабирование: Если выросла нагрузка на модуль аналитики, масштабируется только колоночная БД, а не вся монолитная система.
* Отказоустойчивость: Падение графовой БД сломает блок рекомендаций, но каталог и видеоплеер продолжат работать.

К недостаткам можно отнести:
* Сложность синхронизации (главный минус): Данные могут дублироваться и не хранятся в одном месте (разбросаны по системам). 
* Операционная сложность: Вместо поддержки одной СУБД, команде нужно настраивать, бэкапить и мониторить пять абсолютно разных систем.
* Требования к команде: Инженерам нужно знать синтаксис и особенности работы под капотом для каждой из применяемых моделей.

Задание 4. Выберите конкретные СУБД
На основе ваших решений, предложите конкретные СУБД для каждого модуля и обоснуйте

| Модуль | Конкретная СУБД | Почему именно эту СУБД? |
| :--- | :--- | :--- |
| **Профили пользователей** | **MongoDB** | Самая популярная документоориентированная СУБД. Отлично работает с гибкими JSON-подобными структурами (BSON). Позволяет легко добавлять новые поля в профиль (например, новые привязки соцсетей или кастомные настройки) без сложных миграций схемы. |
| **Прогресс обучения** | **Amazon DynamoDB** | DynamoDB — мощная Key-Value БД, которая обеспечивает стабильный отклик в однозначных миллисекундах при любом масштабе, что критично для постоянного сохранения прогресса. |
| **Поиск по курсам** |  **ClickHouse** | ClickHouse  отлично масштабируется и позволяет быстро фильтровать данные по множеству атрибутов.  *(для классического полнотекстового поиска с опечатками все же стандартом является Elasticsearch).* |
| **Рекомендации** | **Neo4j** | Стандарт для графовых баз данных. Имеет специализированный и интуитивно понятный язык запросов Cypher, который создан специально для обхода узлов и быстрого поиска паттернов. |
| **Кэширование** | **Redis** | In-memory хранилище (работает полностью в оперативной памяти). Обеспечивает микросекундный отклик. Поддерживает TTL (время жизни записи), что идеально подходит для временного хранения сессий и кэширования тяжелых ответов API для снижения нагрузки на основные БД. |
| **Аналитика** | **ClickHouse** | Колоночная OLAP-СУБД, которая способна невероятно быстро выполнять агрегирующие запросы (суммы, средние значения, группировки) на терабайтах данных. Идеальный выбор для построения аналитических дашбордов в реальном времени. |